In [2]:
"""
UNet Cloud Segmentation - Testing & Evaluation Script
=====================================================
Matches the exact data pipeline from U-Net_30_Epoch.ipynb
"""

import os
import numpy as np
import torch
import torch.nn as nn
import rasterio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ─────────────────────────────────────────────
#  CONFIGURATION  ← edit these paths
# ─────────────────────────────────────────────
ROOT          = r"D:\ML\env\notebook\data"          # root data folder
MODEL_PATH    = r"pytorch_unet_cloud_30epochfix_256img.pth"  # path to .pth file
METADATA_CSV  = os.path.join(ROOT, "train_metadata.csv")
FEATURES_DIR  = os.path.join(ROOT, "train_features")
LABELS_DIR    = os.path.join(ROOT, "train_labels")

BANDS         = ["B02", "B03", "B04", "B08"]
IMG_SIZE      = 256
BATCH_SIZE    = 4
THRESHOLD     = 0.5
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PLOTS    = True          # save prediction visualisation images
PLOT_DIR      = "test_plots"  # folder to save visualisation images
NUM_VIS       = 8             # how many samples to visualise

# ─────────────────────────────────────────────
#  MODEL ARCHITECTURE  (must match training)
# ─────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        # ── names MUST match the saved .pth exactly ──
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.c1 = DoubleConv(in_channels, 16)
        self.c2 = DoubleConv(16, 32)
        self.c3 = DoubleConv(32, 64)
        self.c4 = DoubleConv(64, 128)

        self.bn = DoubleConv(128, 256)

        self.c5 = DoubleConv(256 + 128, 128)
        self.c6 = DoubleConv(128 + 64,  64)
        self.c7 = DoubleConv(64  + 32,  32)
        self.c8 = DoubleConv(32  + 16,  16)

        self.out = nn.Sequential(
            nn.Conv2d(16, out_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        c1 = self.c1(x);       p1 = self.pool(c1)
        c2 = self.c2(p1);      p2 = self.pool(c2)
        c3 = self.c3(p2);      p3 = self.pool(c3)
        c4 = self.c4(p3);      p4 = self.pool(c4)

        bn = self.bn(p4)

        u4 = self.up(bn);  u4 = torch.cat([u4, c4], dim=1); c5 = self.c5(u4)
        u3 = self.up(c5);  u3 = torch.cat([u3, c3], dim=1); c6 = self.c6(u3)
        u2 = self.up(c6);  u2 = torch.cat([u2, c2], dim=1); c7 = self.c7(u2)
        u1 = self.up(c7);  u1 = torch.cat([u1, c1], dim=1); c8 = self.c8(u1)

        return self.out(c8)

# ─────────────────────────────────────────────
#  DATASET  (matches notebook exactly)
# ─────────────────────────────────────────────
class CloudDataset(Dataset):
    def __init__(self, chip_ids, features_dir, labels_dir, bands, img_size):
        self.chip_ids     = chip_ids
        self.features_dir = features_dir
        self.labels_dir   = labels_dir
        self.bands        = bands
        self.img_size     = img_size

    def __len__(self):
        return len(self.chip_ids)

    def __getitem__(self, idx):
        chip_id = self.chip_ids[idx]

        # Load each band separately, normalise to [0,1]
        band_arrays = []
        for band in self.bands:
            path = os.path.join(self.features_dir, chip_id, f"{band}.tif")
            with rasterio.open(path) as src:
                arr = src.read(1).astype(np.float32) / 65535.0
            band_arrays.append(arr)

        image = torch.tensor(np.stack(band_arrays, axis=0), dtype=torch.float32)
        image = TF.resize(image, [self.img_size, self.img_size], antialias=True)

        # Load label mask
        label_path = os.path.join(self.labels_dir, f"{chip_id}.tif")
        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)
        label = torch.tensor(label[np.newaxis, ...], dtype=torch.float32)
        label = TF.resize(label, [self.img_size, self.img_size], antialias=True)

        return image, label, chip_id

# ─────────────────────────────────────────────
#  METRICS
# ─────────────────────────────────────────────
def compute_metrics(preds, targets, threshold=0.5):
    preds_bin = (preds > threshold).float()
    eps       = 1e-7

    tp = (preds_bin * targets).sum()
    fp = (preds_bin * (1 - targets)).sum()
    fn = ((1 - preds_bin) * targets).sum()
    tn = ((1 - preds_bin) * (1 - targets)).sum()

    accuracy  = (tp + tn) / (tp + tn + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    iou       = tp / (tp + fp + fn + eps)

    return {
        "accuracy":  accuracy.item(),
        "precision": precision.item(),
        "recall":    recall.item(),
        "iou":       iou.item(),
    }

# ─────────────────────────────────────────────
#  VISUALISATION
# ─────────────────────────────────────────────
def save_visualisations(images, labels, preds, chip_ids, plot_dir, num=8):
    os.makedirs(plot_dir, exist_ok=True)
    n = min(num, len(images))
    fig = plt.figure(figsize=(15, 4 * n))
    gs  = gridspec.GridSpec(n, 4, figure=fig, hspace=0.35, wspace=0.25)

    for i in range(n):
        img   = images[i]           # (4, H, W)  tensor
        label = labels[i, 0].numpy()
        pred  = (preds[i, 0].numpy() > THRESHOLD).astype(np.uint8)
        rgb   = np.stack([img[2], img[1], img[0]], axis=-1)  # B04/B03/B02 as RGB
        rgb   = np.clip(rgb * 3.5, 0, 1)                     # brightness boost

        ax0 = fig.add_subplot(gs[i, 0]); ax0.imshow(rgb);   ax0.set_title(f"RGB  {chip_ids[i]}", fontsize=7); ax0.axis("off")
        ax1 = fig.add_subplot(gs[i, 1]); ax1.imshow(img[3].numpy(), cmap="gray"); ax1.set_title("NIR (B08)", fontsize=7); ax1.axis("off")
        ax2 = fig.add_subplot(gs[i, 2]); ax2.imshow(label, cmap="gray", vmin=0, vmax=1); ax2.set_title("Ground Truth", fontsize=7); ax2.axis("off")
        ax3 = fig.add_subplot(gs[i, 3]); ax3.imshow(pred,  cmap="gray", vmin=0, vmax=1); ax3.set_title("Prediction", fontsize=7); ax3.axis("off")

    plt.savefig(os.path.join(plot_dir, "predictions.png"), dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  Visualisations saved → {os.path.join(plot_dir, 'predictions.png')}")

# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────
def main():
    print(f"\n{'='*55}")
    print("  UNet Cloud Segmentation — Test Evaluation")
    print(f"{'='*55}")
    print(f"  Device : {DEVICE}")
    print(f"  Model  : {MODEL_PATH}")
    print(f"  Data   : {ROOT}\n")

    # ── Reproduce the exact train/val/test split from training ──
    metadata = pd.read_csv(METADATA_CSV)

    # Keep only chips whose feature folder AND label file actually exist
    valid_ids = []
    for cid in metadata["chip_id"].unique():
        feat_ok  = all(os.path.exists(os.path.join(FEATURES_DIR, cid, f"{b}.tif")) for b in BANDS)
        label_ok = os.path.exists(os.path.join(LABELS_DIR, f"{cid}.tif"))
        if feat_ok and label_ok:
            valid_ids.append(cid)

    print(f"  Usable chips: {len(valid_ids)}")

    train_ids, temp_ids = train_test_split(valid_ids, test_size=0.30, random_state=42)
    val_ids,   test_ids = train_test_split(temp_ids,  test_size=0.667, random_state=42)
    print(f"  Split  → train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")

    # Evaluate on the test set (change to val_ids to evaluate on validation set)
    eval_ids = test_ids
    print(f"  Evaluating on TEST set ({len(eval_ids)} chips)\n")

    dataset    = CloudDataset(eval_ids, FEATURES_DIR, LABELS_DIR, BANDS, IMG_SIZE)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ── Load model ──
    model = UNet(in_channels=4, out_channels=1).to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

    # Handle both raw state_dict and wrapped checkpoints
    state = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state)
    model.eval()
    print("  Model loaded successfully.\n")

    # ── Inference loop ──
    all_metrics  = {"accuracy": [], "precision": [], "recall": [], "iou": []}
    vis_images, vis_labels, vis_preds, vis_ids = [], [], [], []

    with torch.no_grad():
        for images, labels, chip_ids in tqdm(dataloader, desc="  Evaluating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            preds  = model(images)

            m = compute_metrics(preds, labels, THRESHOLD)
            for k in all_metrics:
                all_metrics[k].append(m[k])

            # Collect first NUM_VIS samples for visualisation
            if len(vis_images) < NUM_VIS:
                n_take = min(NUM_VIS - len(vis_images), images.size(0))
                vis_images.extend(images[:n_take].cpu())
                vis_labels.extend(labels[:n_take].cpu())
                vis_preds.extend(preds[:n_take].cpu())
                vis_ids.extend(chip_ids[:n_take])

    # ── Print results ──
    print(f"\n{'─'*45}")
    print("  TEST SET RESULTS")
    print(f"{'─'*45}")
    for k, vals in all_metrics.items():
        print(f"  {k.capitalize():12s}: {np.mean(vals):.4f}  (±{np.std(vals):.4f})")
    print(f"{'─'*45}\n")

    # ── Save visualisations ──
    if SAVE_PLOTS:
        vis_images_t = torch.stack(vis_images)
        vis_labels_t = torch.stack(vis_labels)
        vis_preds_t  = torch.stack(vis_preds)
        save_visualisations(vis_images_t, vis_labels_t, vis_preds_t, vis_ids, PLOT_DIR, NUM_VIS)

    print("  Done.\n")


if __name__ == "__main__":
    main()


  UNet Cloud Segmentation — Test Evaluation
  Device : cuda
  Model  : pytorch_unet_cloud_30epochfix_256img.pth
  Data   : D:\ML\env\notebook\data

  Usable chips: 11748
  Split  → train=8223  val=1173  test=2352
  Evaluating on TEST set (2352 chips)

  Model loaded successfully.



  Evaluating: 100%|██████████████████████████████████████████████████████████████████| 588/588 [01:12<00:00,  8.09it/s]



─────────────────────────────────────────────
  TEST SET RESULTS
─────────────────────────────────────────────
  Accuracy    : 0.8853  (±0.0960)
  Precision   : 0.9083  (±0.1200)
  Recall      : 0.9035  (±0.1067)
  Iou         : 0.8264  (±0.1386)
─────────────────────────────────────────────

  Visualisations saved → test_plots\predictions.png
  Done.

